### 1. Initializing Time-MOE and Training ###

In [15]:
import pandas as pd
import numpy as np
import huggingface 
from time_moe.datasets.time_moe_dataset import TimeMoEDataset
from time_moe.datasets.time_moe_window_dataset import TimeMoEWindowDataset
import datetime
import os
import torch
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    

# Configure logging
logging.basicConfig(    
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

print(f"CWD: {os.getcwd()}")


# # Set which GPU(s) are visible
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Use only GPU 1

# # Check what GPUs PyTorch sees
# print(f'Number of GPUs Available: {torch.cuda.device_count()}')  # should show 1 if you set "1"
# print(f'Device 0 Name: {torch.cuda.get_device_name(0)}')  # the GPU PyTorch will use


CWD: c:\Users\yinki\OneDrive\NUS\BT4101\fyp-kiat\ML_Core\src


In [ ]:
# Loading of config
from utils.utils import load_config, config_to_args

config_file_path = '../config/config_tickers_448_7_Channels_with_temporal_tape_v2.yaml'
config_dict = load_config(config_file_path)

place_holders_to_replace = {'version_name': config_dict['regression']['common']['version_name']}
args = config_to_args(config_dict, place_holders_to_replace)

print(f"Current Version Name: {args.regression.common.version_name}")

Current Version Name: tickers_448_7_Channels_with_temporal_tape_v2


### Model Training Stage Time-MOE ###

In [3]:
# from time_moe.datasets import *
# from time_moe.runner import *
# from time_moe.trainer import *
# from time_moe.utils import *

# runner = TimeMoeRunner(
#         model_path=args.regression.Time_MOE.model_path,
#         output_path=args.regression.Time_MOE.output_path,
#         seed=args.regression.Time_MOE.train_seed,
#     )

# train_model_params_dict = args.regression.Time_MOE.train_model_args.__dict__
# train_model_params_dict['args'] = args
    
# model, _ = runner.train_model(
#     **train_model_params_dict
# )

### Time-MoE Hyperparameter Training and Retraining ###


In [4]:
from time_moe.datasets import *
from time_moe.runner import *
from time_moe.trainer import *
from time_moe.utils import *

runner = TimeMoeRunner(
        model_path=args.regression.Time_MOE.model_path,
        output_path=args.regression.Time_MOE.output_path,
        seed=args.regression.Time_MOE.train_seed,
    )

train_model_params_dict = args.regression.Time_MOE.train_model_args.__dict__
optuna_grid = args.regression.Time_MOE.optuna_grid
train_model_params_dict['args'] = args
train_model_params_dict['optuna_grid'] = optuna_grid.__dict__
    
model, best_params = runner.train_model(
    optuna_search=True,
    **train_model_params_dict,
)


# We then update and then do retraining
updated_model_config_update = {'model_configs': {}}
logging.info(f'Retraining on full training set')
for key, value in best_params.hyperparameters.items():
    if key in train_model_params_dict:
        logging.info(f'Updating {key} from {train_model_params_dict[key]} to {value}')
        train_model_params_dict[key] = value
    else:
        logging.info(f'Tracking Model Config Optimal Param for {key}: {value}')
        updated_model_config_update['model_configs'][key] = value

# Retraining with best hyperparameters
model, _ = runner.train_model(
    **train_model_params_dict,
    **updated_model_config_update
)


2025-10-27 17:19:19.637891: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-27 17:19:19.685359: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-27 17:19:20.944584: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-27 17:19:21 [INFO] PyTorch version 2.8.0 available.
2025-10-27 17:19:21 

2025-10-27 17:19:21,687 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Set global_batch_size to 128
2025-10-27 17:19:21,688 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Set micro_batch_size to 16
2025-10-27 17:19:21,689 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Set gradient_accumulation_steps to 8
2025-10-27 17:19:21,690 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Set precision to fp32
2025-10-27 17:19:21,691 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Set normalization to zero
2025-10-27 17:19:21,692 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Use Eager Attention


/home/yinkiat/.pyenv/versions/time-moe-env/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2025-10-27 17:19:22,001 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Channel Configs Inputted: [[63, 1, 1], [6, 1, 4], [6, 1, 5], [10, 1, 1], [5, 1, 1], [5, 1, 1], [5, 1, 2]]


/home/yinkiat/.pyenv/versions/time-moe-env/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(


...

2025-10-27 17:50:21,844 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Total Dataset Size: 24300....
2025-10-27 17:50:21,844 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Using cached dataset (avoiding disk I/O)
2025-10-27 17:50:23,194 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Processing dataset to fixed-size sub-sequences....


2025-10-27 17:50:23 [INFO] MultiFrequencyTimeSeriesDatasetV2 initialized with Price context_length=63, prediction_length=8, window_size=63, stride=1, inferencing_mode=False
2025-10-27 17:50:23 [INFO] Total windows available: 70725


2025-10-27 17:50:23,229 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Total Dataset Size: 70725....
2025-10-27 17:50:23,230 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: Initializaing Model Training full without Optuna Search...
2025-10-27 17:50:23,231 - log_util.py[pid:1134898;line:52:log_in_local_rank_0] - INFO: TimeMoETrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'gradient_accumulation_kwargs': None},
adafactor=False,
adam_beta1=0.9037050239740964,
adam_beta2=0.9675889344224324,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=3003,
dataloader_drop_last=False,
dataloader_num_workers=4,
dataloader_persistent_workers=True,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=False,
ddp_timeout=1800,
deb

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

### TensorBoard Visualisation ###

In [ ]:
### Tensorboard Visualisaiong
import tensorflow as tf
import pandas as pd
from pathlib import Path
import os

# Path to your events file
event_file = Path(f"{args.regression.Time_MOE.output_path}/tb_logs")
logging.info(f"Searching for TB Logs @ {event_file}")
latest_event = [i for i in os.listdir(event_file) if i.startswith('events.out.tfevents')][-1]
logging.info(f"Found TensorBoard Event File {latest_event}")
file_path = os.path.join(event_file, latest_event)

# for f in os.listdir(event_file):
#     if f.startswith('events.out.tfevents'):
#         file_path = os.path.join(event_file, f)
#         logging.info(f"Found TensorBoard Event File {f}")
#         break
        
# Lists to store results
steps = []
tags = []
values = []

# Iterate through the event file
for e in tf.compat.v1.train.summary_iterator(str(file_path)):
    for v in e.summary.value:
        steps.append(e.step)
        tags.append(v.tag)
        values.append(v.simple_value)

# Convert to DataFrame for easy analysis
df = pd.DataFrame({
    "step": steps,
    "tag": tags,
    "value": values
}).drop_duplicates()

# Assume df is your DataFrame
df_filtered = df[df['step'] != 0]  # ignore step 0

# Pivot with aggregation to handle duplicates
df_pivot = df_filtered.pivot_table(
    index='step',
    columns='tag',
    values='value',
    aggfunc='first'  # take the first occurrence if duplicates exist
).reset_index()

df_pivot

In [ ]:
# Plotting of Tensorboard logs in 2x2 subplots
import matplotlib.pyplot as plt

metrics = ['train/loss', 'train/grad_norm', 'train/learning_rate', 'eval/loss']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 2x2 layout
axes = axes.flatten()  # flatten into 1D array for easy looping

for i, m in enumerate(metrics):
    ax = axes[i]
    df_pivot_metrics = df_pivot[['step', m]].dropna()
    ax.plot(df_pivot_metrics['step'], df_pivot_metrics[m], label=m)
    ax.set_title(f'{m}', fontsize=14)
    ax.set_xlabel('Step', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=10)
    ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.show()


#### Pushing to HuggingFace to save Tensor Weights, Optional ####


In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone, timedelta

# Define Singapore timezone (UTC+8)
sgt = timezone(timedelta(hours=8))
datetime_str = datetime.now(sgt).strftime("%Y-%m-%d %H:%M:%S")

# Initialize API
api = HfApi(token=args.regression.Time_MOE.train_model_args.hub_token)

# Upload folder to Hugging Face Hub
api.upload_folder(
    folder_path=args.regression.Time_MOE.output_path,
    repo_id=args.regression.Time_MOE.model_path,
    repo_type="model",
    commit_message=f"{args.regression.common.version_name} model training @ {datetime_str}"
)


### Model Inferencing Stage Time-MoE ###

In [17]:
from S3_Model_Training import S3_model_inferencing_multi_frequency


# Model Inferencing Code
all_sequences_inf = S3_model_inferencing_multi_frequency(args, None, None)

2026-01-23 20:41:35 [INFO] Inititating Model Inferencing....................
2026-01-23 20:41:35 [INFO] Model set as None! Loading from HuggingFace @ kiatkock/MV-Time-MOE
c:\Users\yinki\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2026-01-23 20:41:36,574 - log_util.py[pid:12120;line:52:log_in_local_rank_0] - INFO: Channel Configs Inputted: [[63, 1, 1], [6, 1, 4], [6, 1, 5], [10, 1, 1], [5, 1, 1], [5, 1, 1], [5, 1, 2]]


c:\Users\yinki\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(
2026-01-23 20:43:13 [INFO] MultiFrequencyTimeSeriesDatasetV2 initialized with Price context_length=63, prediction_length=8, window_size=63, stride=1, inferencing_mode=True
2026-01-23 20:43:13 [INFO] Total windows available: 80460
2026-01-23 20:43:13 [INFO] Created DataLoader with 80460 samples, batch_size=8, shuffle=True, num_workers=0
100%|██████████| 10058/10058 [2:09:27<00:00,  1.29it/s] 
2026-01-23 22:52:41 [INFO] Model Inferencing Completed!
2026-01-23 22:52:41 [INFO] Saving the individual inferenced sequences
2026-01-23 22:53:24 [INFO] Saved the individual inferenced sequences to ../data/outputs/tickers_448_7_Channels_with_temporal_tape_v2_all_inference_sequences.pkl
